In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

# Deep Neural Networks

## Session 19a: Tensorflow
### Text Generation using RNN
<img src='../../prasami_images/prasami_color_tutorials_small.png' style = 'width:400px;' alt="By Pramod Sharma : pramod.sharma@prasami.com" align="left"/>

### Import TensorFlow and other libraries

In [ ]:
# Lets import some libraries
from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import tensorflow as tf

In [ ]:
# Some basic parameters

inpDir = Path('../../input') # location where input data is stored
outDir = Path('../output') # location to store outputs
modelDir = Path('../models') # location to store models
subDir = 'text_gen' # location to store models
(modelDir / subDir).mkdir(exist_ok=True, parents=True) # create directory if it does not exist

RANDOM_STATE = 24 # for initialization ----- REMEMBER: to remove at the time of promotion to production
np.random.seed(RANDOM_STATE) # Set Random Seed for reproducible  results
tf.random.set_seed(RANDOM_STATE)

BATCH_SIZE = 64
EPOCHS = 75 # number of cycles to run
ALPHA = 0.001 # learning rate

NUM_GENERATE = 1000 # number of characters to generate
TEMPERATURE = 0.8 # 1.0 # for text generation, to control the randomness of predictions
BUFFER_SIZE = 10000 # for shuffling the dataset

SEQ_LENGTH = 100  
EMBEDDING_DIM = 256
RNN_UNITS = 1024

In [ ]:
physical_devices = tf.config.list_physical_devices('GPU') 

if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

### Shakespeare dataset

In [ ]:
filePath = inpDir / subDir / 'shakespeare.txt'
filePath

In [ ]:
text = open(filePath, 'rb').read().decode(encoding='utf-8')

len(text)

#tf.io.read_file(filePath).numpy()..decode(encoding='utf-8')

In [ ]:
#text

In [ ]:
print(text[:400])

In [ ]:
vocab = sorted(set(text))
len(vocab)

In [ ]:
char2idx = {u:i for i, u in enumerate(vocab)} # 


idx2char = np.array(vocab)

text_as_int = np.array([char2idx[c] for c in text])

text_as_int.shape

In [ ]:
text_as_int

In [ ]:
type(text_as_int)

In [ ]:
text_as_int.shape

In [ ]:
idx2char[47]

In [ ]:
char2idx

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices([1.,2.,3.])

print (list(dataset.as_numpy_iterator()))

In [ ]:
seq_length = 100

example_per_epoch = len(text) // (seq_length+1)

char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

for i in char_dataset.take(10):
    
    print (i.numpy(), '|', idx2char[i.numpy()])

In [ ]:
sequences = char_dataset.batch(seq_length+1, drop_remainder=True)

for item in sequences.take(2):
    
    print (item)

In [ ]:
for item in sequences.take(2):
    
    print (repr( ''.join(idx2char[item.numpy()] ) ) )

In [ ]:
def split_input_target(chunk):
    
    input_text = chunk[:-1]
    
    target_text = chunk[1:]
    
    return input_text, target_text

dataset = sequences.map(split_input_target)

In [ ]:
for inp_ex, tar_ex in dataset.take (2):
    print (repr( ''.join(idx2char[inp_ex.numpy()] ) ))
    print (repr( ''.join(idx2char[tar_ex.numpy()] ) ))
    print ('*'*50, '\n')

In [ ]:
# Shuffle, batch, and prefetch for performance
dataset = dataset.shuffle(buffer_size=BUFFER_SIZE)
dataset = dataset.batch(batch_size=BATCH_SIZE, drop_remainder=True)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
vocab_size = len(vocab)

In [ ]:
def build_model(vocab_size, 
                embedding_dim = EMBEDDING_DIM, 
                rnn_units = RNN_UNITS, 
                batch_size = BATCH_SIZE, 
                stateful=True):
    
    return tf.keras.models.Sequential([
            
        # Input layer specifies the expected shape
        tf.keras.layers.Input(shape=(None,), batch_size=batch_size),  # sequence length is None to allow variable length
        
        tf.keras.layers.Embedding(vocab_size, 
                                  embedding_dim
                                 ),

        # The model will take as input an integer matrix of size (batch,
        # input_length), and the largest integer (i.e. word index) in the input
        # should be no larger than vocab_size (vocabulary size).
        # Now model.output_shape is (None, embedding_dim, input_length), where `None` is the batch
        # dimension.
        
        tf.keras.layers.GRU(units=rnn_units,
                            return_sequences=True, 
                            stateful=stateful, 
                            recurrent_initializer='glorot_uniform'
                           ),
        tf.keras.layers.Dense(vocab_size)
    ])

In [ ]:
# vocab_size, embedding_dim, rnn_units, batch_size
model = build_model(vocab_size= len(vocab))

In [ ]:
for input_ex_batch, target_ex_batch in dataset.take(1):
    ex_batch_pred = model(input_ex_batch)

In [ ]:
ex_batch_pred.shape

In [ ]:
model.summary()

In [ ]:
sampled_indices = tf.random.categorical(ex_batch_pred[0], num_samples=1)
print(sampled_indices.shape)
sampled_indices = tf.squeeze(sampled_indices, axis = -1).numpy()
print(sampled_indices.shape)

display(sampled_indices)
print (repr( ''.join(idx2char[sampled_indices] ) ))

In [ ]:
loss_fn = tf.losses.SparseCategoricalCrossentropy(from_logits=True)

In [ ]:
model.compile(optimizer = 'adam', loss=loss_fn)

In [ ]:
chkPtPath = modelDir / subDir

chkPtPrefix = chkPtPath / 'chkpt_{epoch}.keras'

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(filepath=chkPtPrefix)

In [ ]:
history = model.fit(dataset, epochs= EPOCHS,
                     callbacks=[checkpoint_callback], verbose = 2)

In [ ]:
hist_df = pd.DataFrame(history.history)
hist_df.loc[hist_df['loss'] == hist_df['loss'].min()]


In [ ]:
ax = hist_df.plot()
ax.hlines(hist_df.loc[hist_df['loss'] == hist_df['loss'].min()].values, xmin = 0, xmax= EPOCHS, color = 'g')
ax.vlines(hist_df.loc[hist_df['loss'] == hist_df['loss'].min()].index, 
          ymin = hist_df.loc[hist_df['loss'] == hist_df['loss'].min()].values-0.1, 
          ymax= hist_df.loc[hist_df['loss'] == hist_df['loss'].max()].values-0.1, 
          color = 'g')
ax.grid()

In [ ]:
!ls {chkPtPath}

In [ ]:
hist_df.loc[hist_df['loss'] == hist_df['loss'].min()].index.to_numpy()[0]

In [ ]:
model_num = hist_df.loc[hist_df['loss'] == hist_df['loss'].min()].index.to_numpy()[0]
checkpoint_path = chkPtPath / f'chkpt_{model_num}.keras'
# Step1: create model 
# vocab_size, embedding_dim, rnn_units, batch_size
model1 = build_model(vocab_size= len(vocab), 
                    embedding_dim=EMBEDDING_DIM, 
                    rnn_units = RNN_UNITS,
                    batch_size= 1)

# Step 2: Load only weights from the .keras file
model1.load_weights(checkpoint_path)

model1.build ( tf.TensorShape ( [1, None ] ) )

In [ ]:
model1.summary()

In [ ]:
def generate_text(model, start_string):
    
    num_generate =  1000
    input_eval = [char2idx[s] for s in start_string] # [37, 48, 56 ]
    print (f'Input: {start_string} | {input_eval}\n')
    input_eval = tf.expand_dims(input_eval, 0) # tf.Tensor (1, 1, 5)
    text_generated = []
    
    # model.reset_states()
    
    for i in range(num_generate):
        
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0)
        predictions = predictions / TEMPERATURE # to control the randomness of predictions

        predict_td = tf.random.categorical(predictions, 
                                            num_samples=1)[-1,0].numpy()
        
        input_eval = tf.expand_dims([predict_td], 0)
        text_generated.append(idx2char[predict_td])
        
    return start_string+''.join(text_generated)

In [ ]:
print (generate_text(model1, start_string=u'ROMEO:'))